In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

# Exploratory Data Analysis (EDA)

Exploratory Data Analysis is the most important step in any data project. Before building models or drawing conclusions, you need to **understand your data** — its structure, quality, patterns, and anomalies.

For sensor data, EDA typically answers questions like:
- Are there gaps or outliers in the sensor readings?
- What are the daily and seasonal patterns?
- How do different variables relate to each other (e.g., outdoor temperature vs. heating energy)?
- Has something changed over time (e.g., HVAC system replacement, occupancy changes)?

This chapter walks through a systematic EDA workflow using a real temperature and humidity dataset from four flats.

## EDA Workflow Overview

::::{grid} 2 2 4 4
:gutter: 2

:::{grid-item-card} Step 1
:link: step-1-load-and-inspect
**Load & Inspect**

`shape`, `info()`, `describe()`
:::

:::{grid-item-card} Step 2
:link: step-2-data-quality
**Data Quality**

Missing values, duplicates, gaps
:::

:::{grid-item-card} Step 3
:link: step-3-time-series-overview
**Time Series Overview**

Plot raw data, zoom in
:::

:::{grid-item-card} Step 4
:link: step-4-distribution-analysis
**Distribution Analysis**

Histograms, box plots
:::

:::{grid-item-card} Step 5
:link: step-5-outlier-detection
**Outlier Detection**

IQR, z-score, domain rules
:::

:::{grid-item-card} Step 6
:link: step-6-correlation-analysis
**Correlation Analysis**

Correlation matrix, scatter plots
:::

:::{grid-item-card} Step 7
:link: step-7-seasonal-decomposition
**Seasonal Decomposition**

Trend, seasonal, residual
:::

:::{grid-item-card} Step 8
:link: step-8-autocorrelation
**Autocorrelation**

Dominant cycles & periodicities
:::

::::

In [ ]:
import pandas as pd
import numpy as np

(step-1-load-and-inspect)=
## Step 1: Load and Inspect

Always start by loading the data, parsing dates, and getting a first overview. If you are unfamiliar with loading data in pandas, see the [Loading Data](loadingData) chapter first.

In [ ]:
url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv"

df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time")

print(f"Shape: {df.shape}")
print(f"Period: {df.index.min()} to {df.index.max()}")
print(f"Duration: {df.index.max() - df.index.min()}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

:::{admonition} What to look for in describe()
:class: tip dropdown
- **count** differs between columns? → some columns have more missing values
- **min/max** outside physically plausible range? → possible sensor error (e.g., temperature of -999 or 99)
- **mean vs. 50%** (median) very different? → skewed data, possibly caused by outliers
- **std** very large relative to mean? → high variability or mixed data
:::

(step-2-data-quality)=
## Step 2: Data Quality

Real-world sensor data almost always has quality issues: missing values, duplicate timestamps, or irregular intervals. Identifying these early prevents wrong conclusions later.

In [ ]:
# Missing values per column
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "% missing": (df.isna().mean() * 100).round(1)
})
missing

In [ ]:
from pyedautils.data_quality import plot_missing_values_heatmap

fig = plot_missing_values_heatmap(df)
fig.show()

:::{admonition} Interpreting the missing values heatmap
:class: tip dropdown
Red bands indicate periods where sensors reported no data. Common causes:
- **Vertical red band across all sensors** → system-wide outage (network, gateway, power)
- **Single sensor red band** → individual sensor failure or battery empty
- **Red at the end** → sensors stopped reporting (decommissioned or data export cutoff)
:::

In [ ]:
# Check for duplicate timestamps and irregular intervals
print(f"Duplicate timestamps: {df.index.duplicated().sum()}")
print(f"Index is monotonic:  {df.index.is_monotonic_increasing}")

# Most common interval
time_diffs = df.index.to_series().diff().dropna()
print(f"Most common interval: {time_diffs.mode()[0]}")
print(f"Min interval: {time_diffs.min()}")
print(f"Max interval: {time_diffs.max()}")

(step-3-time-series-overview)=
## Step 3: Time Series Overview

Plot the raw data to get a visual overview. Interactive plots allow zooming into specific periods.

In [ ]:
from pyedautils.plots.stat import plot_timeseries

fig = plot_timeseries(df, columns=["FlatA_Temp"], title="FlatA Temperature", ylab="Temperature [°C]")
fig.show()

# Column list reused in later cells
temp_cols = [col for col in df.columns if "Temp" in col]

In [ ]:
fig = plot_timeseries(df, columns=temp_cols, title="All Flats — Temperature", ylab="Temperature [°C]")
fig.show()

(step-4-distribution-analysis)=
## Step 4: Distribution Analysis

Histograms and box plots reveal the **distribution** of values — how often each temperature occurs, whether there are unusual clusters, and how the distribution changes over time.

In [ ]:
from pyedautils.plots.stat import plot_distribution

fig = plot_distribution(df, columns=temp_cols, title="Temperature Distribution by Flat", xlab="Temperature [°C]")
fig.show()

In [ ]:
from pyedautils.plots.stat import plot_boxplot

fig = plot_boxplot(df, column="FlatA_Temp", groupby="month", title="FlatA Temperature by Month", ylab="Temperature [°C]")
fig.show()

In [ ]:
fig = plot_boxplot(df, column="FlatA_Temp", groupby="hour", title="FlatA Temperature by Hour of Day", ylab="Temperature [°C]")
fig.show()

:::{admonition} What distribution patterns tell you
:class: tip dropdown
- **Bimodal histogram** (two peaks) → different operating modes (e.g., day/night, occupied/unoccupied)
- **Box plot outliers by month** → seasonal anomalies or sensor issues in specific periods
- **Wider boxes in summer** → higher temperature variability (no heating control)
- **Flat line across hours** → constant temperature (heavily controlled environment or stuck sensor)
:::

(step-5-outlier-detection)=
## Step 5: Outlier Detection

Outliers in sensor data can be real events (window opening, sensor splash) or measurement errors (sensor drift, communication glitch). Both are important to identify.

In [ ]:
from pyedautils.data_quality import calc_outliers

result = calc_outliers(df, column="FlatA_Temp")
print(f"IQR range: [{result['lower']:.1f}, {result['upper']:.1f}] °C")
print(f"Outliers: {result['count']} of {len(df)} ({result['percentage']}%)")

In [ ]:
from pyedautils.plots.stat import plot_outliers

fig = plot_outliers(df, column="FlatA_Temp", title="FlatA Temperature — Outlier Detection (IQR)", ylab="Temperature [°C]")
fig.show()

:::{admonition} Outlier detection methods
:class: tip dropdown

| Method | How it works | Best for |
|---|---|---|
| **IQR** | Values outside [Q1−1.5×IQR, Q3+1.5×IQR] | General purpose, robust to distribution shape |
| **Z-score** | Values where \|z\| > 3 | Normally distributed data |
| **Domain rules** | e.g., temperature outside [5°C, 45°C] | When you know the physical limits |
| **Rolling Z-score** | Z-score within a rolling window | Detecting local anomalies in seasonal data |

**Important:** Not every statistical outlier is a data error! A sudden temperature drop might be a real window opening event. Always investigate before removing outliers.
:::

(step-6-correlation-analysis)=
## Step 6: Correlation Analysis

Understanding how variables relate to each other is key for building data. Do all flats behave similarly? Does temperature correlate with humidity?

In [ ]:
from pyedautils.plots.stat import plot_correlation

fig = plot_correlation(df)
fig.show()

In [ ]:
from pyedautils.plots.stat import plot_scatter

fig = plot_scatter(df, x_column="FlatA_Temp", y_column="FlatA_Hum",
                   title="FlatA: Temperature vs. Humidity",
                   xlab="Temperature [°C]", ylab="Humidity [%]")
fig.show()

:::{admonition} Reading correlation in building data
:class: tip dropdown
- **High positive correlation between flats** → similar orientation, insulation, or shared heating system
- **Negative correlation temperature ↔ humidity** → physically expected (warmer air holds more moisture, relative humidity drops)
- **Low correlation between flats** → different usage patterns, orientation, or occupancy
:::

(step-7-seasonal-decomposition)=
## Step 7: Seasonal Decomposition

Seasonal decomposition separates a time series into three components:
- **Trend** — the long-term direction (is the building getting warmer over the years?)
- **Seasonal** — the repeating pattern (daily cycle, annual cycle)
- **Residual** — what's left after removing trend and seasonality (noise, anomalies, events)

In [ ]:
from pyedautils.plots.stat import plot_decomposition

ts = df[["FlatA_Temp"]].interpolate(limit=6).dropna()
ts = ts.reset_index()
ts.columns = ["timestamp", "value"]

fig = plot_decomposition(ts, period=24, title="Time Series Decomposition — FlatA Temperature", ylab="Temperature [°C]")
fig.show()

:::{admonition} Understanding the decomposition
:class: tip dropdown
- **Trend** shows the long-term temperature evolution — in residential buildings, this often follows outdoor temperature with a delay (building thermal mass)
- **Seasonal** shows the repeating daily pattern — temperature rises during the day and drops at night
- **Residual** shows everything unexplained — large spikes here indicate anomalies (window openings, sensor errors, unusual events)

**STL vs. classical decomposition:** STL (Seasonal and Trend decomposition using Loess) is more robust to outliers and handles changing seasonal patterns. Use `robust=True` for sensor data, which often contains spikes.

**Choosing the period:**

| Data frequency | Daily cycle | Weekly cycle |
|---|---|---|
| 15-minute | period=96 | period=672 |
| Hourly | period=24 | period=168 |
| Daily | period=365 | — |
:::

(step-8-autocorrelation)=
## Step 8: Autocorrelation

Autocorrelation shows how a signal correlates with itself at different time lags. Peaks in the autocorrelation plot reveal the dominant periodicities in the data.

In [ ]:
from pyedautils.plots.stat import plot_autocorrelation

fig = plot_autocorrelation(df, column="FlatA_Temp", lags=168, title="Autocorrelation — FlatA Temperature")
fig.show()

:::{admonition} Reading the autocorrelation plot
:class: tip dropdown
- **Peak at lag 24** → daily cycle (value now is similar to value 24 hours ago)
- **Peak at lag 168** → weekly cycle (similar behavior same day last week)
- **Slow decay** → strong trend in the data (non-stationary)
- **Values inside the blue band** → not statistically significant (noise)

For building data, you almost always see a strong daily cycle. Weekly cycles appear in commercial buildings (weekday/weekend) but are less pronounced in residential buildings.
:::

## EDA Checklist

Use this checklist as a starting point for every new dataset:

| Step | Question | Key functions |
|---|---|---|
| **1. Inspect** | How big? What columns? What types? | `shape`, `dtypes`, `info()`, `describe()` |
| **2. Quality** | Missing values? Duplicates? Gaps? | `isna().sum()`, `duplicated()`, `index.diff()` |
| **3. Visualize** | What does the raw data look like? | `px.line()` with range slider |
| **4. Distribute** | How are values distributed? | `px.histogram()`, `px.box()` by hour/month |
| **5. Outliers** | Are there suspicious values? | IQR method, domain rules |
| **6. Correlate** | How do variables relate? | `df.corr()`, `px.scatter()` |
| **7. Decompose** | What are the trend and seasonal patterns? | `STL()`, `seasonal_decompose()` |
| **8. Autocorrelation** | What are the dominant cycles? | `plot_acf()` |